<a href="https://colab.research.google.com/github/Prateekpkini/Intent_and_trajectory/blob/main/Intent_and_trajectory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the nuScenes development kit
!pip install nuscenes-devkit

# 2. Mount Google Drive to access your downloaded nuScenes data
from google.colab import drive
drive.mount('/content/drive')

# 3. Define the path where you stored the 'v1.0-mini' or 'v1.0-trainval' data
# Structure should be: /data/sets/nuscenes/samples, /metadata, etc.
DATAROOT = '/content/drive/MyDrive/nuScenes_Data'

import tarfile
import os

# Define your paths
tgz_path = '/content/drive/MyDrive/nuScenes_Data/v1.0-mini.tgz'
extract_path = '/content/drive/MyDrive/nuScenes_Data'

# Create directory if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# Extracting the file
with tarfile.open(tgz_path, 'r:gz') as tar:
    tar.extractall(path=extract_path)

print("Extraction complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_3926/3073409610.py:25: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete!


In [ ]:
from nuscenes import NuScenes
from nuscenes.prediction import PredictHelper
from nuscenes.eval.prediction.splits import get_prediction_challenge_split

# Load the dataset (using 'v1.0-mini' for initial code testing)
nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=True)
helper = PredictHelper(nusc)

def get_mini_split(nusc):
    # We'll just take all instances in the mini set that have a future
    tokens = []
    for scene in nusc.scene:
        first_sample_token = scene['first_sample_token']
        # Use helper to find instances in this scene
        # This is a quick way to get valid testing tokens
        sample = nusc.get('sample', first_sample_token)
        for ann_token in sample['anns']:
            ann = nusc.get('sample_annotation', ann_token)
            # Format: {instance_token}_{sample_token}
            tokens.append(f"{ann['instance_token']}_{first_sample_token}")
    return tokens

train_tokens = get_mini_split(nusc)
print(f"Loaded {len(train_tokens)} tokens from mini-split.")

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.705 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.
Loaded 328 tokens from mini-split.


In [ ]:
import numpy as np
import torch

def get_agent_features(token):
    instance_token, sample_token = token.split("_")

    # 1. Past trajectory (2s @ 2Hz = 5 points including current)
    past_xy = helper.get_past_for_agent(instance_token, sample_token, seconds=2, in_agent_frame=True)

    # 2. Future Ground Truth (6s @ 2Hz = 12 points)
    future_xy = helper.get_future_for_agent(instance_token, sample_token, seconds=6, in_agent_frame=True)

    # 3. Kinematic Features
    vel = helper.get_velocity_for_agent(instance_token, sample_token)
    accel = helper.get_acceleration_for_agent(instance_token, sample_token)
    yaw_rate = helper.get_heading_change_rate_for_agent(instance_token, sample_token)

    # Handle potential NaNs in kinematics
    kinematics = np.nan_to_num(np.array([vel, accel, yaw_rate]))

    return torch.FloatTensor(past_xy), torch.FloatTensor(future_xy), torch.FloatTensor(kinematics)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50

class MTPModel(nn.Module):
    def __init__(self, num_modes=25, future_steps=12):
        super().__init__()
        # Visual Stream: ResNet50 for Raster Maps
        backbone = resnet50(pretrained=False)
        self.visual_encoder = nn.Sequential(*list(backbone.children())[:-1])

        # Temporal Stream: LSTM for past coordinates
        self.lstm = nn.LSTM(input_size=2, hidden_size=128, batch_first=True)

        # Fusion and Prediction Head
        # ResNet50 output (2048) + LSTM output (128)
        combined_dim = 2048 + 128
        self.reg_head = nn.Linear(combined_dim, num_modes * future_steps * 2)
        self.cls_head = nn.Linear(combined_dim, num_modes)

        self.num_modes = num_modes
        self.future_steps = future_steps

    def forward(self, map_raster, past_traj):
        # 1. Extract visual features
        vis_feat = self.visual_encoder(map_raster).flatten(1)

        # 2. Extract temporal features
        _, (h_n, _) = self.lstm(past_traj)
        temp_feat = h_n.squeeze(0)

        # 3. Concatenate and Predict
        fused = torch.cat([vis_feat, temp_feat], dim=1)

        trajectories = self.reg_head(fused).view(-1, self.num_modes, self.future_steps, 2)
        logits = self.cls_head(fused)

        return trajectories, torch.softmax(logits, dim=1)

In [ ]:
def mtp_loss(pred_traj, pred_probs, labels, alpha=1.0):
    # labels shape: [batch, 12, 2]
    # pred_traj shape: [batch, 25, 12, 2]

    batch_size = pred_traj.size(0)
    labels_exp = labels.unsqueeze(1) # [batch, 1, 12, 2]

    # Calculate Average Displacement Error for each mode
    distances = torch.norm(pred_traj - labels_exp, dim=3)
    ade = distances.mean(dim=2) # [batch, 25]

    # Find the "Winner" mode
    best_mode_idx = torch.argmin(ade, dim=1)

    # Regression Loss on the winner
    best_pred = pred_traj[torch.arange(batch_size), best_mode_idx]
    reg_loss = nn.functional.smooth_l1_loss(best_pred, labels)

    # Classification Loss to prefer the winner
    cls_loss = nn.functional.cross_entropy(pred_probs, best_mode_idx)

    return reg_loss + (alpha * cls_loss)